QUESTÃO III - OBSERVAÇÕES:
Peguei o dataset no Kaggle. link: [link do kaggle]
Disponibilizo os modelos (VGG, RESNET50, MOBILEV2) para download no meu google drive: [link do drive]
Professor, infelizmente não consegui pensar em nada pra possibilitar um link online com os arquivos do dataset, para que sua execução pudesse ser feita sem ter que baixar ou configurar nada. O Sr. pode fazer o download do dataset fácilmente pelos links acima, caso queira executar o train e também as validações precisamos do dataset, até para re-executar as validações rodando os modelos que estão no .pth de cada um, os modelos estão arquivo ZIP que disponibilizei no link e também no arquivo classroom, algoritmos que precisam destes:
`task04_q3a_train_SVM.ipynb`
`task04_q3b_train_models.ipynb`
`task04_q3ab_run_models.ipynb`

**QUESTÃO III - A**  
Utilizei o **HOG do OpenCV** como extrator de características com isso analisei a distribuição das direções dos gradientes nas imagens, capturando informações sobre padrões de bordas e texturas que são boas características para diferenciar gatos de cachorros, arestas das orelhas por exemplo, formato. Todas as imagens passei pra 128x128 e usei um **SVM Linear** (LinearSVC) que é um classificador tradicional que tenta encontrar um hiperplano que separe melhor as classes (grupos). Por questões de desempenho no meu notebook local, reduzi o conjunto de dados para 0.3 do split total. Mesmo assim, até que consegui alguns resultados. O pipeline ficou: carrega imagens, redimensiona, extrai HOG, treina SVM e avalia resultados.

**QUESTÃO III - B**  
Utilizei como solicitado no enunciado da questão transferência de aprendizado com **VGG16**, **ResNet50** e **MobileNetV2** com a lib PyTorch. Só uma obs, eu tentei gerar um modelo manual, na verdade consegui rodar no meu notebook localmente, mas todas as vezes que testei não passava das 40 epochs, pois o meu notebook fechava tudo, killava. Com 37 epochs que foi até onde chegou e pegou Acc0.95. Então criei uma instância na ec2 e gerei 3 modelos, os quais vão ser executados aqui na questão B. Para agilizar usei o pytorch que já tem pesos pré-treinados em ImageNet, salvo engano. Então daqui me bastou definir apenas a última camada pra poder classificar entre gatos e cachorros, isso porque as arquiteturas pré definidas do pytorch são muito boas, tive ótimos resultado em poucas epochs, total de cada foram 10, mas os melhores modelos com 0.99 foram com menos de 7 inclusive. São bem otimizadas, isso reflete nos resultados, poucas epochs temos altos índices de acurácia e precisão, loss constante. Como disse ali, apliquei uma pequena lógica pra salvar o melhor resultado, as vezes quando não usamos o stop pra pouca diferença no loss e validação, eu vou salvando o melhor modelo e pronto. Se nenhum bater, aquele é.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import joblib
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchinfo import summary
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
from torchvision.models import vgg16, resnet50, mobilenet_v2

from PIL import UnidentifiedImageError

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
df_dir = '/Users/luryand/Documents/VC/img/task04/PetImages'
models_dir = '/Users/luryand/Documents/VC/output/task04'

cat_files = os.listdir(os.path.join(df_dir, 'Cat'))
dog_files = os.listdir(os.path.join(df_dir, 'Dog'))

In [ ]:
# Transformações para modelos de deep learning
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def safe_loader(path):
    try:
        return Image.open(path).convert('RGB')
    except (UnidentifiedImageError, OSError, ValueError) as e:
        print(f"Erro ao abrir {path}: {e}")
        return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

# Split dataset
full_dataset = ImageFolder(df_dir, transform=None)
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Data loaders para os dl models
val_dataset_dl = ImageFolder(df_dir, loader=safe_loader, transform=transform)
val_sampler = SubsetRandomSampler(val_indices.indices)
val_loader = DataLoader(
    val_dataset_dl, batch_size=32, sampler=val_sampler, num_workers=0
)

print(f"Classes: {val_dataset_dl.classes}")
print(f"Total de amostras: {len(full_dataset)}")
print(f"Amostras de treino: {len(train_indices)}")
print(f"Amostras de validação: {len(val_indices)}")

In [ ]:
def extract_hog(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (128, 128))
    hog_descriptor = cv2.HOGDescriptor()
    hog_descriptor.compute(img).flatten()

def visualize_prediction_grid(model_name, class_names, images=None, labels=None, preds=None, 
                             img_paths=None, is_svm=False):
    fig, axes = plt.subplots(7, 7, figsize=(14, 14))
    fig.suptitle(f"{model_name}: Prediction Examples (Green: Correct, Red: Incorrect)", fontsize=16)

    axes = axes.flatten()
    
    if is_svm:
        samples = list(zip(img_paths, labels, preds))
        np.random.shuffle(samples)
        samples = samples[:49]
        
        for i, (path, label, pred) in enumerate(samples):
            if i >= 49:
                break
                
            img = cv2.imread(path)
            if img is None:
                img = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))
            
            ax = axes[i]
            ax.imshow(img)
            
            correct = label == pred
            color = 'green' if correct else 'red'
            
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)
            
            ax.set_title(f"True: {class_names[label]}\nPred: {class_names[pred]}", 
                        color=color, fontsize=8)
            ax.axis('off')
    else:
        all_samples = []
        
        for i in range(len(images)):
            all_samples.append((images[i], labels[i], preds[i]))
            
        np.random.shuffle(all_samples)
        samples = all_samples[:49]
        
        for i, (img, label, pred) in enumerate(samples):
            if i >= 49:
                break
                
            ax = axes[i]
            
            img = img * 0.5 + 0.5
            ax.imshow(img.permute(1, 2, 0).numpy())
            
            correct = label == pred
            color = 'green' if correct else 'red'
            
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)
            
            ax.set_title(f"True: {class_names[label]}\nPred: {class_names[pred]}", 
                        color=color, fontsize=8)
            ax.axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.subplots_adjust(hspace=0.3)
    plt.show()

def evaluate_svm(model_path):
    print("\nEval HOG+SVM Model")
    
    clf = joblib.load(model_path)
    
    val_paths, val_labels = [], []
    for idx in val_indices.indices:
        path, label = full_dataset.samples[idx]
        val_paths.append(path)
        val_labels.append(label)
    
    X_val = []
    valid_indices = []
    
    for i, path in enumerate(tqdm(val_paths, desc="Extracting HOG features")):
        hog_features = extract_hog(path)
        if hog_features is not None:
            X_val.append(hog_features)
            valid_indices.append(i)
    
    X_val = np.array(X_val)
    y_val = np.array([val_labels[i] for i in valid_indices])
    val_paths_filtered = [val_paths[i] for i in valid_indices]
    
    # Predict
    y_pred = clf.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    cm = confusion_matrix(y_val, y_pred)
    
    class_counts = {}
    for class_idx, class_name in enumerate(full_dataset.classes):
        true_count = np.sum(y_val == class_idx)
        correct_count = np.sum((y_val == class_idx) & (y_pred == class_idx))
        class_counts[class_name] = (correct_count, true_count)

    print(f"HOG+SVM Acc: {acc:.4f}")
    for class_name, (correct, total) in class_counts.items():
        print(f"  • {class_name}: {correct}/{total} correct ({correct/total*100:.2f}%)")

    visualize_prediction_grid("HOG+SVM", full_dataset.classes, 
                             img_paths=val_paths_filtered, 
                             labels=y_val, preds=y_pred, is_svm=True)
    
    return acc, cm, full_dataset.classes

def evaluate_dl_model(model_type, model_path):
    print(f"\nEval {model_type} Model")
    
    # Initialize model architecture based on type
    if model_type == "VGG16":
        model = vgg16(pretrained=False)
        model.classifier[6] = nn.Linear(4096, 2)
    elif model_type == "ResNet50":
        model = resnet50(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif model_type == "MobileNetV2":
        model = mobilenet_v2(pretrained=False)
        model.classifier[1] = nn.Linear(model.last_channel, 2)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_images = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Evaluating {model_type}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            all_images.extend(images.cpu())
            
            if len(all_images) >= 100:
                break
    
    acc = correct / total
    cm = confusion_matrix(all_labels[:total], all_preds[:total])
    
    class_counts = {}
    for class_idx, class_name in enumerate(val_dataset_dl.classes):
        true_count = np.sum(np.array(all_labels[:total]) == class_idx)
        correct_count = np.sum((np.array(all_labels[:total]) == class_idx) & 
                              (np.array(all_preds[:total]) == class_idx))
        class_counts[class_name] = (correct_count, true_count)
    
    print(f"{model_type} Acc: {acc:.4f}")
    for class_name, (correct, total) in class_counts.items():
        print(f"  • {class_name}: {correct}/{total} corretas ({correct/total*100:.2f}%)")
    
    visualize_prediction_grid(model_type, val_dataset_dl.classes, 
                             images=all_images[:100], 
                             labels=all_labels[:100], 
                             preds=all_preds[:100], is_svm=False)
    
    return acc, cm, val_dataset_dl.classes

In [ ]:
results = {}
cms = {}
class_names = {}

# 1. Eval HOG+SVM
svm_path = os.path.join(models_dir, 'hog_svm_model.joblib')
acc, cm, names = evaluate_svm(svm_path)
results['HOG+SVM'] = acc
cms['HOG+SVM'] = cm
class_names['HOG+SVM'] = names

# 2. Eval VGG16
vgg16_path = os.path.join(models_dir, 'modelo_catsdogs_vgg16.pth')
acc, cm, names = evaluate_dl_model("VGG16", vgg16_path)
results['VGG16'] = acc
cms['VGG16'] = cm
class_names['VGG16'] = names

# 3. Eval ResNet50
resnet50_path = os.path.join(models_dir, 'modelo_catsdogs_resnet50.pth')
acc, cm, names = evaluate_dl_model("ResNet50", resnet50_path)
results['ResNet50'] = acc
cms['ResNet50'] = cm
class_names['ResNet50'] = names

# 4. Eval MobileNetV2
mobilenet_path = os.path.join(models_dir, 'modelo_catsdogs_mobilenetv2.pth')
acc, cm, names = evaluate_dl_model("MobileNetV2", mobilenet_path)
results['MobileNetV2'] = acc
cms['MobileNetV2'] = cm
class_names['MobileNetV2'] = names

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

if cms:
    num_models = len(cms)
    fig, axes = plt.subplots(1, num_models, figsize=(5*num_models, 4))
    
    for i, (model_name, cm) in enumerate(cms.items()):
        if num_models == 1:
            ax = axes
        else:
            ax = axes[i]
            
        ConfusionMatrixDisplay(cm, display_labels=class_names[model_name]).plot(ax=ax, values_format='d', colorbar=False)
        ax.set_title(f"{model_name}\nAccuracy: {results[model_name]:.4f}")
        
        ax.tick_params(axis='both', which='major', labelsize=8)
        
    plt.tight_layout()
    plt.show()